# 🏭 Üretim Anomali Tespiti — XGBoost Forecasting
## Benchmarking: Isolation Forest ile Kıyaslamalı Analiz
> **Veri:** `excel_partiler_total.xlsx` &nbsp;|&nbsp; **7 Sensör (S1–S7)** &nbsp;|&nbsp; **12 Parti (6N + 6A)**  
> **Yaklaşım:** Normal partilerle eğitilmiş MultiOutputRegressor(XGBoost) → Reconstruction Error → Anomali Skoru

### Notebook Akışı
1. Kurulum & İçe Aktarma
2. Veri Yükleme & Temizleme
3–5. EDA (Genel Bakış · Zaman Serileri · KDE)
6. Sliding Window Feature/Target Extraction
7. Eğitim / Test Ayrımı
8. Model Eğitimi (MultiOutputRegressor + XGBoost)
9. Anomali Skoru (MAE Tabanlı)
10. Eşik Optimizasyonu
11. Performans Metrikleri (ROC · PR · CM)
12. Anomali Skor Zaman Serileri
13. SHAP Açıklanabilirlik (TreeExplainer)
14. Sensör Bazında Residual Analizi
15. Özet Tablo & Dosya İndirme

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   ÜRETİM ANOMALİ TESPİTİ — XGBoost Forecasting Tabanlı                 ║
# ║   Benchmarking: Isolation Forest ile kıyaslamalı analiz                 ║
# ║   Veri: excel_partiler_total.xlsx  |  12 Parti  |  7 Sensör             ║
# ╚══════════════════════════════════════════════════════════════════════════╝


### HÜCRE 1: Kurulum & İçe Aktarma

In [ ]:
# ── HÜCRE 1: Kurulum & İçe Aktarma ─────────────────────────────────────────

!pip install -q openpyxl xgboost shap scikit-learn matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import shap
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score,
    mean_absolute_error, mean_squared_error,
)
import os, json
from google.colab import files

# ── Görsel stil (IF ile birebir aynı) ──
plt.rcParams.update({
    "figure.facecolor": "#0F1117",
    "axes.facecolor":   "#1A1D27",
    "axes.edgecolor":   "#3A3F54",
    "axes.labelcolor":  "#C8CDD8",
    "xtick.color":      "#8891A5",
    "ytick.color":      "#8891A5",
    "text.color":       "#E2E5EE",
    "grid.color":       "#2A2F42",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
    "axes.titlesize":   11,
    "axes.titleweight": "bold",
    "axes.titlecolor":  "#FFFFFF",
    "figure.dpi":       130,
})

PALETTE_NORMAL = "#3DD68C"
PALETTE_ANOM   = "#FF5370"
PALETTE_SCORE  = "#7C83FD"
PALETTE_THRESH = "#FFB547"

print("✓ Kurulum ve içe aktarma tamamlandı.")


### HÜCRE 2: Veri Yükleme & Temizleme

In [ ]:
# ── HÜCRE 2: Veri Yükleme & Temizleme ──────────────────────────────────────

uploaded = files.upload()   # excel_partiler_total.xlsx seç
FILENAME = list(uploaded.keys())[0]

df_raw = pd.read_excel(FILENAME, sheet_name=0)

# ── VERİ TEMİZLEME ──────────────────────────────────────────────────────────
# Sensör sütunlarını sayısala zorla; kirli değerleri (örn. '"') NaN yap
_SENSOR_RAW = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
for _col in _SENSOR_RAW:
    if _col in df_raw.columns:
        df_raw[_col] = pd.to_numeric(df_raw[_col], errors="coerce")

df_raw[_SENSOR_RAW] = df_raw[_SENSOR_RAW].ffill().fillna(0)
print(f"✓ Veri temizlendi — kalan NaN: {df_raw[_SENSOR_RAW].isna().sum().sum()}")

# ── Sabitler ────────────────────────────────────────────────────────────────
SENSOR_COLS = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
SENSOR_LABELS = [
    "AK Sıcaklığı", "BK1 Sıcaklığı", "BK2 Sıcaklık",
    "AK Seviyesi",  "İletkenlik",     "BK1 Seviyesi",
    "BK2 Seviyesi"
]
# S4 ölçek sorunu — diğer sensörlerden ayrı gösterilecek
SENSOR_COLS_NO_S4  = [s for s in SENSOR_COLS  if s  != "S4_AK_Seviyesi"]
SENSOR_LABELS_NO_S4= [l for l in SENSOR_LABELS if l != "AK Seviyesi"]

WINDOW_SIZE = 30   # IF ile aynı

# Etiket: Kusur sütunu (0=normal, 1=anormal)
df_raw["label"] = df_raw["Kusur"].astype(int)

batch_labels = (
    df_raw.groupby("batch_id")["label"]
    .first()
    .reset_index()
)
batch_labels["durum"] = batch_labels["label"].map({0: "Normal", 1: "Anormal"})

print(f"✓ Veri yüklendi: {len(df_raw):,} satır | {df_raw['batch_id'].nunique()} parti")
print(f"  Normal  : {(batch_labels['label']==0).sum()} parti → eğitim")
print(f"  Anormal : {(batch_labels['label']==1).sum()} parti → test")
print()
print(batch_labels.to_string(index=False))


### HÜCRE 3: EDA — Genel Bakış (S4 ayrı subplot)

In [ ]:
# ── HÜCRE 3: EDA — Genel Bakış (S4 ayrı subplot) ────────────────────────────

fig = plt.figure(figsize=(20, 14))
fig.suptitle("EDA — Parti Genel Bakış  [XGBoost Notebook]",
             fontsize=14, color="#FFFFFF", y=1.01)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.32)

# 3a: Parti uzunlukları
ax1 = fig.add_subplot(gs[0, 0])
lengths = df_raw.groupby("batch_id")["t"].max().reset_index()
lengths = lengths.merge(batch_labels, on="batch_id")
colors_bar = [PALETTE_ANOM if l == 1 else PALETTE_NORMAL
              for l in lengths["label"]]
bars = ax1.barh(lengths["batch_id"].astype(str), lengths["t"],
                color=colors_bar, alpha=0.85, edgecolor="none")
ax1.axvline(lengths["t"].mean(), color=PALETTE_THRESH, ls="--", lw=1.2,
            label=f"Ort. {lengths['t'].mean():.0f} dk")
for bar, val in zip(bars, lengths["t"]):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             str(val), va="center", fontsize=7, color="#C8CDD8")
ax1.set_xlim(0, lengths["t"].max() * 1.18)
ax1.set_xlabel("Ölçüm Sayısı (dakika)")
ax1.set_title("Parti Uzunlukları")
ax1.legend(fontsize=7)
legend_els = [mpatches.Patch(color=PALETTE_NORMAL, label="Normal"),
              mpatches.Patch(color=PALETTE_ANOM,   label="Anormal")]
ax1.legend(handles=legend_els + [
    Line2D([0],[0], color=PALETTE_THRESH, ls="--", lw=1.2,
           label=f"Ort. {lengths['t'].mean():.0f} dk")
], fontsize=7)

# 3b: Boxplot — S4 HARİÇ diğer 6 sensör
ax2 = fig.add_subplot(gs[0, 1])
melt_no_s4 = df_raw[SENSOR_COLS_NO_S4 + ["label"]].copy()
melt_no_s4.columns = SENSOR_LABELS_NO_S4 + ["label"]
melt_long_no_s4 = melt_no_s4.melt(
    id_vars="label", var_name="Sensör", value_name="Değer"
)
melt_long_no_s4["Durum"] = melt_long_no_s4["label"].map({0: "Normal", 1: "Anormal"})
sns.boxplot(
    data=melt_long_no_s4, x="Sensör", y="Değer", hue="Durum",
    palette={"Normal": PALETTE_NORMAL, "Anormal": PALETTE_ANOM},
    ax=ax2, linewidth=0.7, fliersize=1.5, width=0.6,
    boxprops=dict(alpha=0.85),
)
ax2.set_title("Sensör Dağılımı: Normal vs Anormal  (S4 hariç)")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=25)
ax2.legend(fontsize=7)

# 3c: Boxplot — Sadece S4 (AK Seviyesi)
ax3 = fig.add_subplot(gs[1, 0])
melt_s4 = df_raw[["S4_AK_Seviyesi", "label"]].copy()
melt_s4["Durum"] = melt_s4["label"].map({0: "Normal", 1: "Anormal"})
sns.boxplot(
    data=melt_s4, x="Durum", y="S4_AK_Seviyesi", hue="Durum",
    palette={"Normal": PALETTE_NORMAL, "Anormal": PALETTE_ANOM},
    ax=ax3, linewidth=0.7, fliersize=1.5, width=0.45,
    boxprops=dict(alpha=0.85), legend=False,
)
ax3.set_title("AK Seviyesi (S4) — Ayrı Ölçek")
ax3.set_xlabel("")
ax3.set_ylabel("AK Seviyesi")

# 3d: Sensör ortalamaları — S4 HARİÇ
ax4 = fig.add_subplot(gs[1, 1])
normal_mean  = df_raw[df_raw["label"]==0][SENSOR_COLS_NO_S4].mean()
anom_mean    = df_raw[df_raw["label"]==1][SENSOR_COLS_NO_S4].mean()
x = np.arange(len(SENSOR_LABELS_NO_S4))
w = 0.35
ax4.bar(x - w/2, normal_mean, w, label="Normal",
        color=PALETTE_NORMAL, alpha=0.85, edgecolor="none")
ax4.bar(x + w/2, anom_mean,   w, label="Anormal",
        color=PALETTE_ANOM,   alpha=0.85, edgecolor="none")
ax4.set_xticks(x)
ax4.set_xticklabels(SENSOR_LABELS_NO_S4, rotation=25, fontsize=8)
ax4.set_title("Sensör Ortalamaları: Normal vs Anormal  (S4 hariç)")
ax4.legend(fontsize=8)
ax4.set_ylabel("Ortalama Değer")

# 3e: AK Seviyesi ortalama — ayrı bar
ax5 = fig.add_subplot(gs[2, 0])
s4_means = {
    "Normal" : df_raw[df_raw["label"]==0]["S4_AK_Seviyesi"].mean(),
    "Anormal": df_raw[df_raw["label"]==1]["S4_AK_Seviyesi"].mean(),
}
ax5.bar(["Normal", "Anormal"], list(s4_means.values()),
        color=[PALETTE_NORMAL, PALETTE_ANOM], alpha=0.85,
        edgecolor="none", width=0.4)
for i, (k, v) in enumerate(s4_means.items()):
    ax5.text(i, v + 30, f"{v:.0f}", ha="center", fontsize=9, color="#E2E5EE")
ax5.set_title("AK Seviyesi (S4) — Ortalama  (Ayrı Ölçek)")
ax5.set_ylabel("Ortalama AK Seviyesi")

# 3f: Korelasyon matrisi (normal partiler)
ax6 = fig.add_subplot(gs[2, 1])
corr = df_raw[df_raw["label"]==0][SENSOR_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, ax=ax6, mask=mask, cmap=cmap, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            linewidths=0.4, square=True,
            xticklabels=SENSOR_LABELS, yticklabels=SENSOR_LABELS,
            cbar_kws={"shrink": 0.7})
ax6.set_title("Sensör Korelasyonu (Normal Partiler)")
ax6.tick_params(axis="x", rotation=30, labelsize=6)
ax6.tick_params(axis="y", rotation=0,  labelsize=6)

plt.savefig("eda_genel_bakis_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_genel_bakis_xgb.png")
print("✓ EDA Genel Bakış indirildi.")


### HÜCRE 4: EDA — Sensör Zaman Serileri

In [ ]:
# ── HÜCRE 4: EDA — Sensör Zaman Serileri ────────────────────────────────────

fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle("EDA — Sensör Zaman Serileri (Tüm Partiler)",
             fontsize=13, color="#FFFFFF")
axes = axes.flatten()

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    for _, row in batch_labels.iterrows():
        bid   = row["batch_id"]
        lbl   = row["label"]
        bdata = df_raw[df_raw["batch_id"] == bid].sort_values("t")
        color = PALETTE_ANOM if lbl == 1 else PALETTE_NORMAL
        ax.plot(bdata["t"], bdata[sensor],
                color=color, alpha=0.55 if lbl==1 else 0.70,
                lw=1.0 if lbl==1 else 0.8)
    ax.set_title(label_tr)
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("Değer", fontsize=7)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(handles=[
            Line2D([0],[0], color=PALETTE_NORMAL, lw=2, label="Normal"),
            Line2D([0],[0], color=PALETTE_ANOM,   lw=2, label="Anormal"),
        ], fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_zaman_serileri_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_zaman_serileri_xgb.png")
print("✓ Zaman serileri grafiği indirildi.")


### HÜCRE 5: EDA — KDE Dağılımları

In [ ]:
# ── HÜCRE 5: EDA — KDE Dağılımları ─────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("EDA — Sensör Dağılımları (KDE)", fontsize=13, color="#FFFFFF")
axes = axes.flatten()

normal_data = df_raw[df_raw["label"] == 0]
anom_data   = df_raw[df_raw["label"] == 1]

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    sns.kdeplot(normal_data[sensor].dropna(), ax=ax,
                color=PALETTE_NORMAL, fill=True, alpha=0.35, lw=1.5, label="Normal")
    sns.kdeplot(anom_data[sensor].dropna(),   ax=ax,
                color=PALETTE_ANOM,   fill=True, alpha=0.35, lw=1.5, label="Anormal")
    ax.set_title(label_tr, fontsize=9)
    ax.set_xlabel("Değer", fontsize=7)
    ax.set_ylabel("Yoğunluk", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_kde_dagilimlar_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_kde_dagilimlar_xgb.png")
print("✓ KDE dağılım grafiği indirildi.")

print("\n── SENSÖR İSTATİSTİKLERİ (Normal) ──")
print(normal_data[SENSOR_COLS].describe().round(2).to_string())
print("\n── SENSÖR İSTATİSTİKLERİ (Anormal) ──")
print(anom_data[SENSOR_COLS].describe().round(2).to_string())


### HÜCRE 6: Sliding Window Feature & Target Extraction

In [ ]:
# ── HÜCRE 6: Sliding Window Feature & Target Extraction ─────────────────────
#
# Görev: t anındaki 7 sensörü tahmin et (hedef = Y)
#        Girdi (X) = [t-30, t-29, ..., t-1] arasındaki 7×30 = 210 feature
# Bu konfigürasyon IF ile birebir aynı window yapısını kullanır.

def extract_xy(df, window_size=WINDOW_SIZE):
    """
    X : (t-window_size) ... (t-1) arasındaki sensör değerleri → flatten → 210 feat
    Y : t anındaki 7 sensör değeri
    meta: batch_id, t, label
    """
    X_list, Y_list, meta_list = [], [], []

    for batch_id, group in df.groupby("batch_id"):
        group   = group.sort_values("t").reset_index(drop=True)
        sensors = group[SENSOR_COLS].values   # (n, 7)
        label   = group["label"].iloc[0]
        n       = len(sensors)

        # t=window_size'dan itibaren: X=[t-30..t-1], Y=t
        for t_idx in range(window_size, n):
            X_window = sensors[t_idx - window_size : t_idx].flatten()  # (210,)
            Y_target = sensors[t_idx]                                   # (7,)
            X_list.append(X_window)
            Y_list.append(Y_target)
            meta_list.append({
                "batch_id": batch_id,
                "t":        group["t"].iloc[t_idx],
                "label":    label,
            })

    X    = np.vstack(X_list)
    Y    = np.vstack(Y_list)
    meta = pd.DataFrame(meta_list).reset_index(drop=True)

    n_feat = window_size * len(SENSOR_COLS)
    feat_names = [
        f"w{i // len(SENSOR_COLS)}_S{(i % len(SENSOR_COLS)) + 1}"
        for i in range(n_feat)
    ]
    print(f"✓ Feature extraction tamamlandı")
    print(f"  Window size    : {window_size} dk")
    print(f"  Feature sayısı : {n_feat} (X) → 7 hedef (Y)")
    print(f"  Toplam örnek   : {len(X):,}")
    return X, Y, meta, feat_names


X_all, Y_all, meta_all, feat_names = extract_xy(df_raw)


### HÜCRE 7: Eğitim / Test Ayrımı

In [ ]:
# ── HÜCRE 7: Eğitim / Test Ayrımı ───────────────────────────────────────────

normal_batch_ids = batch_labels[batch_labels["label"]==0]["batch_id"].tolist()
anom_batch_ids   = batch_labels[batch_labels["label"]==1]["batch_id"].tolist()

train_mask = meta_all["batch_id"].isin(normal_batch_ids).values

X_train = X_all[train_mask]
Y_train = Y_all[train_mask]
X_test  = X_all          # 12 parti — tümü test
Y_test  = Y_all
meta_test = meta_all.copy()

print(f"✓ Eğitim seti : {len(X_train):,} örnek — {len(normal_batch_ids)} normal parti")
print(f"✓ Test seti   : {len(X_test):,}  örnek — 12 parti (6N + 6A)")

# Ölçekleme — XGBoost için zorunlu değil ama tutarlılık için yapıyoruz
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
Y_train_sc = scaler_Y.fit_transform(Y_train)
X_test_sc  = scaler_X.transform(X_test)


### HÜCRE 8: Model Eğitimi

In [ ]:
# ── HÜCRE 8: Model Eğitimi ───────────────────────────────────────────────────
# MultiOutputRegressor(XGBRegressor) → 7 sensörü aynı anda tahmin eder
# Her sensör için ayrı bir XGBoost modeli fit edilir (toplam 7 model)

base_xgb = XGBRegressor(
    n_estimators      = 300,
    max_depth         = 5,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0,
)

model = MultiOutputRegressor(base_xgb, n_jobs=-1)
model.fit(X_train_sc, Y_train_sc)

# Test setinde tahmin → gerçek ölçeğe geri döndür
Y_pred_sc = model.predict(X_test_sc)
Y_pred    = scaler_Y.inverse_transform(Y_pred_sc)
Y_actual  = Y_test

print(f"✓ XGBoost modeli eğitildi")
print(f"  Toplam alt model : {len(model.estimators_)} (7 sensör)")
print(f"  Her model ağacı  : 300")
print(f"  Eğitim örneği    : {len(X_train):,}")

# Eğitim seti reconstruction error (baseline)
Y_train_pred_sc = model.predict(X_train_sc)
Y_train_pred    = scaler_Y.inverse_transform(Y_train_pred_sc)
train_mae_per_sensor = np.abs(Y_train - Y_train_pred).mean(axis=0)
print(f"\n── Eğitim MAE (sensör bazında) ──")
for s, mae in zip(SENSOR_LABELS, train_mae_per_sensor):
    print(f"  {s:18s}: {mae:.4f}")


### HÜCRE 9: Anomali Skoru Hesaplama

In [ ]:
# ── HÜCRE 9: Anomali Skoru Hesaplama ────────────────────────────────────────
# Anomali skoru = zaman noktası bazında 7 sensörün MAE ortalaması
# (Reconstruction Error — tahmin ile gerçek arasındaki fark)

residuals   = np.abs(Y_actual - Y_pred)          # (n, 7) — her sensör için hata
anom_scores = residuals.mean(axis=1)              # (n,)   — ortalama MAE skoru

meta_test = meta_test.copy()
meta_test["anomaly_score"] = anom_scores
# Sensör bazında residual'ı da sakla (SHAP için)
for i, s in enumerate(SENSOR_COLS):
    meta_test[f"resid_{s}"] = residuals[:, i]

print(f"✓ Anomali skoru hesaplandı (MAE tabanlı)")
print(f"  Genel min   : {anom_scores.min():.4f}")
print(f"  Genel max   : {anom_scores.max():.4f}")
print(f"  Normal ort  : {anom_scores[meta_test['label']==0].mean():.4f}")
print(f"  Anormal ort : {anom_scores[meta_test['label']==1].mean():.4f}")


### HÜCRE 10: Eşik Optimizasyonu

In [ ]:
# ── HÜCRE 10: Eşik Optimizasyonu ────────────────────────────────────────────
# IF ile birebir aynı mantık: parti düzeyi max skor → F1 maksimum eşik

def batch_level_eval(meta_df, threshold):
    grp = meta_df.groupby("batch_id").agg(
        true_label = ("label",         "first"),
        max_score  = ("anomaly_score", "max"),
        mean_score = ("anomaly_score", "mean"),
    ).reset_index()
    grp["pred_label"] = (grp["max_score"] > threshold).astype(int)
    return grp

thresholds = np.percentile(anom_scores, np.arange(70, 99, 0.5))
thr_results = []
for thr in thresholds:
    grp = batch_level_eval(meta_test, thr)
    try:
        auc = roc_auc_score(grp["true_label"], grp["max_score"])
    except Exception:
        auc = 0.0
    tp = ((grp["pred_label"]==1) & (grp["true_label"]==1)).sum()
    fp = ((grp["pred_label"]==1) & (grp["true_label"]==0)).sum()
    fn = ((grp["pred_label"]==0) & (grp["true_label"]==1)).sum()
    f1 = tp / (tp + 0.5*(fp + fn) + 1e-9)
    thr_results.append({"threshold": thr, "f1": f1, "auc": auc,
                        "tp": tp, "fp": fp, "fn": fn})

thr_df = pd.DataFrame(thr_results)
best_row = thr_df.loc[thr_df["f1"].idxmax()]
BEST_THRESHOLD = best_row["threshold"]

print(f"✓ Eşik optimizasyonu tamamlandı")
print(f"  En iyi eşik : {BEST_THRESHOLD:.4f}")
print(f"  F1 skoru    : {best_row['f1']:.3f}")
print(f"  ROC-AUC     : {best_row['auc']:.3f}")
print(f"  TP: {int(best_row['tp'])}  FP: {int(best_row['fp'])}  FN: {int(best_row['fn'])}")


### HÜCRE 11: Performans Metrikleri & Görselleştirme

In [ ]:
# ── HÜCRE 11: Performans Metrikleri & Görselleştirme ────────────────────────

batch_results = batch_level_eval(meta_test, BEST_THRESHOLD)

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Performans Metrikleri — XGBoost Forecasting",
             fontsize=14, color="#FFFFFF", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 11a: Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(batch_results["true_label"], batch_results["pred_label"])
sns.heatmap(cm, annot=True, fmt="d", ax=ax1, cmap="YlOrRd",
            xticklabels=["Normal","Anormal"],
            yticklabels=["Normal","Anormal"],
            annot_kws={"size": 16, "weight": "bold"},
            linewidths=0.5, cbar=False)
ax1.set_title("Confusion Matrix (Parti Düzeyi)")
ax1.set_xlabel("Tahmin", fontsize=9)
ax1.set_ylabel("Gerçek",  fontsize=9)

# 11b: ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(batch_results["true_label"], batch_results["max_score"])
auc_val = roc_auc_score(batch_results["true_label"], batch_results["max_score"])
ax2.plot(fpr, tpr, color=PALETTE_SCORE, lw=2, label=f"ROC AUC = {auc_val:.3f}")
ax2.plot([0,1],[0,1], "w--", lw=0.8, alpha=0.5)
ax2.fill_between(fpr, tpr, alpha=0.15, color=PALETTE_SCORE)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Eğrisi (Parti Düzeyi)")
ax2.legend(fontsize=9)
ax2.set_xlim([-0.02, 1.02]); ax2.set_ylim([-0.02, 1.05])

# 11c: Precision-Recall
ax3 = fig.add_subplot(gs[0, 2])
prec, rec, _ = precision_recall_curve(
    batch_results["true_label"], batch_results["max_score"])
ap = average_precision_score(batch_results["true_label"], batch_results["max_score"])
ax3.plot(rec, prec, color=PALETTE_ANOM, lw=2, label=f"AP = {ap:.3f}")
ax3.fill_between(rec, prec, alpha=0.15, color=PALETTE_ANOM)
ax3.set_xlabel("Recall"); ax3.set_ylabel("Precision")
ax3.set_title("Precision-Recall Eğrisi")
ax3.legend(fontsize=9)
ax3.set_xlim([-0.02, 1.02]); ax3.set_ylim([-0.02, 1.05])

# 11d: Eşik vs F1
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(thr_df["threshold"], thr_df["f1"], color=PALETTE_SCORE, lw=1.8)
ax4.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5,
            label=f"Best = {BEST_THRESHOLD:.4f}")
ax4.set_xlabel("Eşik (MAE)"); ax4.set_ylabel("F1 Skoru")
ax4.set_title("Eşik Optimizasyonu")
ax4.legend(fontsize=8)

# 11e: Parti bazında max anomali skoru
ax5 = fig.add_subplot(gs[1, 1])
br = batch_results.sort_values("max_score", ascending=True)
colors_br = [PALETTE_ANOM if l==1 else PALETTE_NORMAL for l in br["true_label"]]
ax5.barh(br["batch_id"].astype(str), br["max_score"],
         color=colors_br, alpha=0.85, edgecolor="none")
ax5.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5)
ax5.set_xlabel("Max Anomali Skoru (MAE)")
ax5.set_title("Parti Bazında Max Skor")
ax5.legend(handles=[
    mpatches.Patch(color=PALETTE_NORMAL, label="Normal"),
    mpatches.Patch(color=PALETTE_ANOM,   label="Anormal"),
    Line2D([0],[0], color=PALETTE_THRESH, ls="--", lw=1.5,
           label=f"Eşik={BEST_THRESHOLD:.4f}")
], fontsize=7)

# 11f: Skor dağılımı
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(meta_test[meta_test["label"]==0]["anomaly_score"], bins=60,
         color=PALETTE_NORMAL, alpha=0.6, density=True,
         label="Normal", edgecolor="none")
ax6.hist(meta_test[meta_test["label"]==1]["anomaly_score"], bins=60,
         color=PALETTE_ANOM,   alpha=0.6, density=True,
         label="Anormal", edgecolor="none")
ax6.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5, label="Eşik")
ax6.set_xlabel("Anomali Skoru (MAE)")
ax6.set_ylabel("Yoğunluk")
ax6.set_title("Skor Dağılımı: Normal vs Anormal")
ax6.legend(fontsize=8)

plt.savefig("performans_metrikleri_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("performans_metrikleri_xgb.png")
print("✓ Performans metrikleri grafiği indirildi.")

print("\n" + "═"*55)
print("  PERFORMANS RAPORU — PARTİ DÜZEYİ  [XGBoost]")
print("═"*55)
print(classification_report(
    batch_results["true_label"], batch_results["pred_label"],
    target_names=["Normal","Anormal"], zero_division=0,
))
print(f"ROC-AUC        : {auc_val:.4f}")
print(f"Avg Precision  : {ap:.4f}")
print(f"Kullanılan Eşik: {BEST_THRESHOLD:.4f}")


### HÜCRE 12: Anomali Skor Zaman Serileri

In [ ]:
# ── HÜCRE 12: Anomali Skor Zaman Serileri ───────────────────────────────────

batches_sorted = (
    batch_results.sort_values(["true_label","batch_id"])["batch_id"].tolist()
)
n_cols = 3
n_rows = int(np.ceil(len(batches_sorted) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4))
fig.suptitle("Anomali Skoru Zaman Serileri — XGBoost MAE",
             fontsize=14, color="#FFFFFF")
axes = axes.flatten()

for i, bid in enumerate(batches_sorted):
    ax   = axes[i]
    bdat = meta_test[meta_test["batch_id"]==bid].sort_values("t")
    lbl  = batch_results.loc[batch_results["batch_id"]==bid, "true_label"].iloc[0]
    pred = batch_results.loc[batch_results["batch_id"]==bid, "pred_label"].iloc[0]
    color = PALETTE_ANOM if lbl==1 else PALETTE_NORMAL

    ax.fill_between(bdat["t"], bdat["anomaly_score"], alpha=0.18, color=color)
    ax.plot(bdat["t"], bdat["anomaly_score"], color=PALETTE_SCORE, lw=1.2, alpha=0.9)
    ax.axhline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.2,
               label=f"Eşik {BEST_THRESHOLD:.3f}")

    breach = bdat[bdat["anomaly_score"] > BEST_THRESHOLD]
    if not breach.empty:
        first_t = breach["t"].iloc[0]
        ax.axvline(first_t, color="#FF9F43", ls=":", lw=1.4,
                   label=f"İlk alarm: t={first_t}")

    status = "✓" if lbl==pred else "✗"
    title_color = "#3DD68C" if lbl==pred else "#FF5370"
    ax.set_title(f"{bid}  [{'ANORMAL' if lbl==1 else 'NORMAL'}]  {status}",
                 fontsize=9, color=title_color)
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("MAE Skoru", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=6)

for j in range(len(batches_sorted), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("anomali_skor_zaman_serileri_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("anomali_skor_zaman_serileri_xgb.png")
print("✓ Anomali skor zaman serileri indirildi.")


### HÜCRE 13: SHAP Açıklanabilirlik (TreeExplainer)

In [ ]:
# ── HÜCRE 13: SHAP Açıklanabilirlik (TreeExplainer) ─────────────────────────
#
# Strateji:
#   Her sensör modeli için TreeExplainer çalıştırıyoruz (7 ayrı XGB).
#   Test noktasında SHAP değeri → o sensörün hatasına hangi giriş feature'larının
#   katkıda bulunduğunu gösterir.
#   Sonra: anormal partilerdeki yüksek-hata anlarındaki SHAP'ı sensör bazında
#   toplayarak "hangi sensör anomaliyi tetikledi?" sorusunu yanıtlıyoruz.

print("SHAP hesaplanıyor (≈2-4 dk, 7 model × test seti)...")

# Anormal partilerdeki eşik üstü satırları al (SHAP örneklemi)
anom_batch_ids_list = batch_labels[batch_labels["label"]==1]["batch_id"].tolist()
high_error_mask = (
    meta_test["batch_id"].isin(anom_batch_ids_list) &
    (meta_test["anomaly_score"] > BEST_THRESHOLD)
).values

# Çok büyük olmasın diye maks 500 satır örnekle
sample_idx = np.where(high_error_mask)[0]
if len(sample_idx) > 500:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(sample_idx, 500, replace=False)

X_shap_sample = X_test_sc[sample_idx]

# Her sensör için SHAP değerlerini hesapla, sensör düzeyinde özetleme yap
# SHAP feature önem → penceredeki her feature'ın katkısı
# Biz bunu sensör grubuna göre toplayarak "hangi sensör önemli" sorusuna cevap veririz

shap_per_sensor_model = {}   # sensor_name → mean(|shap|) array (n_features,)

for s_idx, (s_col, s_label) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    estimator = model.estimators_[s_idx]
    explainer  = shap.TreeExplainer(estimator)
    shap_vals  = explainer.shap_values(X_shap_sample)       # (n_samples, 210)
    shap_mean  = np.abs(shap_vals).mean(axis=0)              # (210,)
    shap_per_sensor_model[s_label] = shap_mean
    print(f"  ✓ {s_label} SHAP tamamlandı")

# Sensör grubuna göre özetleme
# feat_idx i → w = i // 7, s = i % 7 → hangi sensör grubundan geldiği
n_sensors = len(SENSOR_COLS)
grouped_shap = np.zeros((n_sensors, n_sensors))   # [hedef_sensör, girdi_sensör]

for tgt_idx, s_label in enumerate(SENSOR_LABELS):
    shap_arr = shap_per_sensor_model[s_label]     # (210,)
    for src_idx in range(n_sensors):
        # Bu girdi sensörüne ait tüm window feature'larının SHAP toplamı
        feat_indices = [w * n_sensors + src_idx for w in range(WINDOW_SIZE)]
        grouped_shap[tgt_idx, src_idx] = shap_arr[feat_indices].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("SHAP Açıklanabilirlik — XGBoost TreeExplainer\n"
             "(Anormal Partiler, Eşik Üstü Noktalar)",
             fontsize=12, color="#FFFFFF")

# 13a: Girdi sensörü → Toplam SHAP etkisi (tüm hedef sensörlere katkısı)
ax1 = axes[0]
input_sensor_importance = grouped_shap.sum(axis=0)   # her girdi sensörünün toplam etkisi
sorted_idx = np.argsort(input_sensor_importance)
colors_shap = plt.cm.YlOrRd(
    np.linspace(0.3, 0.95, len(SENSOR_LABELS))
)
bars = ax1.barh(
    [SENSOR_LABELS[i] for i in sorted_idx],
    input_sensor_importance[sorted_idx],
    color=colors_shap, alpha=0.9, edgecolor="none"
)
ax1.set_xlabel("Toplam SHAP Katkısı (tüm hedef sensörlere)")
ax1.set_title("Hangi Girdi Sensörü Anomaliyi Tetikliyor?")
for bar, val in zip(bars, input_sensor_importance[sorted_idx]):
    ax1.text(bar.get_width() + 0.0002,
             bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=8, color="#C8CDD8")

# 13b: Isı haritası — hedef sensör × girdi sensörü SHAP matrisi
ax2 = axes[1]
cmap_shap = sns.color_palette("YlOrRd", as_cmap=True)
sns.heatmap(
    grouped_shap,
    ax=ax2,
    xticklabels=SENSOR_LABELS,
    yticklabels=SENSOR_LABELS,
    cmap=cmap_shap,
    annot=True, fmt=".3f", annot_kws={"size": 7},
    linewidths=0.3,
    cbar_kws={"shrink": 0.8, "label": "SHAP Değeri"},
)
ax2.set_xlabel("Girdi Sensörü (window'dan gelen)", fontsize=8)
ax2.set_ylabel("Hedef Sensör (tahmin edilen)", fontsize=8)
ax2.set_title("SHAP Matrisi: Girdi → Hedef Sensör Etkisi")
ax2.tick_params(axis="x", rotation=30, labelsize=7)
ax2.tick_params(axis="y", rotation=0,  labelsize=7)

plt.tight_layout()
plt.savefig("shap_aciklanabilirlik_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("shap_aciklanabilirlik_xgb.png")
print("\n✓ SHAP grafiği indirildi.")

# En önemli girdi sensörü
top_sensor_idx = np.argmax(input_sensor_importance)
print(f"\n  → Anomali sırasında en etkili girdi sensörü: "
      f"[{SENSOR_LABELS[top_sensor_idx]}]")


### HÜCRE 14: Sensör Bazında Residual Analizi

In [ ]:
# ── HÜCRE 14: Sensör Bazında Residual Analizi ────────────────────────────────
# Anormal partilerde hangi sensörün hatası en yüksek?

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Sensör Bazında Residual (Tahmin Hatası) — Anormal Partiler",
             fontsize=13, color="#FFFFFF")
axes = axes.flatten()

anom_meta = meta_test[meta_test["batch_id"].isin(anom_batch_ids_list)]

for i, (s_col, s_label) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    for bid, grp in anom_meta.groupby("batch_id"):
        grp_s = grp.sort_values("t")
        ax.plot(grp_s["t"], grp_s[f"resid_{s_col}"],
                lw=0.9, alpha=0.75, label=str(bid))
    ax.axhline(0, color="#8891A5", ls="--", lw=0.7)
    ax.set_title(s_label, fontsize=9)
    ax.set_xlabel("t", fontsize=7)
    ax.set_ylabel("Residual (|gerçek - tahmin|)", fontsize=7)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=6, ncol=2)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("residual_analizi_xgb.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("residual_analizi_xgb.png")
print("✓ Residual analizi grafiği indirildi.")


### HÜCRE 15: Özet Tablo & Tüm Dosyaları İndir

In [ ]:
# ── HÜCRE 15: Özet Tablo & Tüm Dosyaları İndir ──────────────────────────────

summary_df = batch_results.copy()
summary_df["Durum"]       = summary_df["true_label"].map({0:"Normal",1:"Anormal"})
summary_df["Tahmin"]      = summary_df["pred_label"].map({0:"Normal",1:"Anormal"})
summary_df["Doğru_mu"]    = summary_df["true_label"] == summary_df["pred_label"]
summary_df["İlk_Alarm_t"] = summary_df["batch_id"].apply(
    lambda bid: (
        int(meta_test.loc[
            (meta_test["batch_id"]==bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD),
            "t"
        ].min())
        if not meta_test.loc[
            (meta_test["batch_id"]==bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD), "t"
        ].empty
        else None
    )
)
summary_df = summary_df[[
    "batch_id","Durum","Tahmin","Doğru_mu","max_score","mean_score","İlk_Alarm_t"
]].rename(columns={"batch_id":"Parti","max_score":"Max_MAE","mean_score":"Ort_MAE"})

print("═"*58)
print("  PARTİ BAZINDA SONUÇ ÖZETİ  [XGBoost Forecasting]")
print("═"*58)
print(summary_df.to_string(index=False))

# Dosyaları kaydet
meta_test.to_csv("anomali_skorlari_tam_xgb.csv",
                 index=False, encoding="utf-8-sig")
summary_df.to_csv("parti_sonuc_ozeti_xgb.csv",
                  index=False, encoding="utf-8-sig")

model_params = {
    "model"          : "XGBoost MultiOutputRegressor",
    "window_size"    : WINDOW_SIZE,
    "n_estimators"   : 300,
    "anomaly_score"  : "MAE (mean absolute reconstruction error)",
    "best_threshold" : round(float(BEST_THRESHOLD), 6),
    "roc_auc"        : round(float(auc_val), 4),
    "avg_precision"  : round(float(ap),      4),
    "n_train_samples": int(len(X_train)),
    "n_sensors"      : len(SENSOR_COLS),
    "sensor_names"   : SENSOR_LABELS,
    "top_shap_sensor": SENSOR_LABELS[int(np.argmax(input_sensor_importance))],
}
with open("model_parametreleri_xgb.json", "w", encoding="utf-8") as f:
    json.dump(model_params, f, ensure_ascii=False, indent=2)

files.download("anomali_skorlari_tam_xgb.csv")
files.download("parti_sonuc_ozeti_xgb.csv")
files.download("model_parametreleri_xgb.json")
print("\n✓ Tüm CSV ve JSON dosyaları indirildi.")

print("\n" + "═"*58)
print("  TÜM ADIMLAR TAMAMLANDI ✓  [XGBoost Forecasting]")
print("═"*58)
print("  İndirilen dosyalar:")
print("    📊 eda_genel_bakis_xgb.png")
print("    📈 eda_zaman_serileri_xgb.png")
print("    📉 eda_kde_dagilimlar_xgb.png")
print("    🎯 performans_metrikleri_xgb.png")
print("    ⏱  anomali_skor_zaman_serileri_xgb.png")
print("    🔍 shap_aciklanabilirlik_xgb.png")
print("    📐 residual_analizi_xgb.png")
print("    💾 anomali_skorlari_tam_xgb.csv")
print("    📋 parti_sonuc_ozeti_xgb.csv")
print("    ⚙️  model_parametreleri_xgb.json")
